In [6]:
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, MCSamples
import matplotlib.ticker as ticker
import os

output_folder = '/home/steven/steven_thesis/thesis_config/config/tester/outputs/'


In [49]:
class postprocess :
    def __init__ (self, chain_files) :
        self.chain_files = chain_files
        self.mc = {}
        self.sample_names_dict = {}
        self.sample_values_dict = {}
        # Defining labels for plots
        label_dict = {
                    "om": r"\Omega_m",
                    "sigma_8": r"\sigma_8",
                    "s8": r"s_8",
                    "Sigma_8": r"\Sigma_8",
                    "h0": r"H_0",
                    "omch2": r"\Omega_c h^2",
                    "ombh2": r"\Omega_b h^2",
                    "weight": "weight",
                    "prior": "prior",
                    "like": "likelihood",
                    "post": "posterior",
                    }
        self.alpha_dict = {}

        for key in chain_files :
            sample_names = []
            sample_values = []
            file = chain_files[key]
            with open(file, "r") as self.f:
                data = np.loadtxt(file)
                header = self.f.readline().strip().split()
                non_params = ['prior','like','post','weight']
                # Constructing the array of sample names and sample values based on the output file
                for i,name in enumerate(header) :
                    if name not in non_params :
                        _, param = name.split("--")
                        if param == "s_8_input" : # Somehow s_8_input doesn't work as a name in GetDist.
                            param = "s8"
                        sample_names.append(param)
                        sample_values.append(data[:,i])
                    else :
                        param = name
                        sample_names.append(param)
                        sample_values.append(data[:,i])
                        
                # Preparation for Sigma_8
                omch2 = self.get_param_values(sample_names, sample_values,'omch2')
                ombh2 = self.get_param_values(sample_names, sample_values,'ombh2')
                s8 = self.get_param_values(sample_names, sample_values,'s8')
                h0 = self.get_param_values(sample_names, sample_values,'h0')
                om = (omch2 + ombh2) / h0**2
                sigma_8 = s8 / (om / 0.3)**0.5
                ### Fitting for alpha by performing linear regression : log(sigma_8) = -alpha*log(omega_m/0.3) + c
                if np.asarray(om).size>1 and np.asarray(sigma_8).size>1 :
                    X = np.log(om/0.3)
                    Y = np.log(sigma_8)
                    coeffs, cov = np.polyfit(X,Y,deg=1,cov=True,w=sample_values[index_weight])
                    alpha,Sigma_8_reg = coeffs
                    alpha = -alpha
                    alpha_err , Sigma_8_err = np.sqrt(np.diag(cov))
                    # print(f"alpha = {self.alpha} +/- {self.alpha_err}")
                    # print(f"Sigma_8 (from regression) = {np.exp(Sigma_8_reg)} +/- {Sigma_8_err}")
                    Sigma_8 = sigma_8*(om/0.3)**alpha
                    self.alpha_dict[key] = [alpha, alpha_err]
                    
                    # Extra parameters derived from the chain
                    sample_names.append('Sigma_8')
                    sample_values.append(Sigma_8)     
                    sample_names.append('om')
                    sample_values.append(om)
                    sample_names.append('sigma_8')
                    sample_values.append(sigma_8)

                index_weight = sample_names.index("weight")
                self.sample_names_dict[key] = sample_names
                self.sample_values_dict[key] = sample_values
            
            labels = [label_dict.get(name, name) for name in sample_names]
            self.mc[key] = MCSamples(samples=sample_values, 
                         weights=sample_values[index_weight], 
                         names=sample_names, 
                         labels=labels, 
                         label=key)


    def get_param_values (self, sample_names : list, sample_values : list, param : str) :
        index = sample_names.index(param) if param in sample_names else None
        self.f.seek(0)
        if index is not None :
            return sample_values[index]
        else :
            while True :
                line = self.f.readline().strip().split()
                if param == 's8':
                    param = 's_8_input'
                if param in line :
                    try:
                        return float(line[-1])
                    except ValueError :
                        pass
                if not line :
                    print("end of file reached without finding parameter:", param)
                    return None

    def plot_contour (self, x_param='om', wanted_params = None, wanted_chain = None, dashed_chain = None, colors = None, legend = None) :
        """ 
        wanted_chain : list of keys to plot
        wanted_params : list of parameters to plot against omega_m
        color : list of colors for each chain
        legend : legend showing chains, should be ordered by keys
        """
        if dashed_chain is None :
            dashed_chain = []
        elif not isinstance(dashed_chain, str) :
            raise TypeError("Only a single chain can be dashed. If you have provided a list of one chain, please provide it as a string instead.")
        else :
            wanted_mc_dashed = self.mc[dashed_chain]
        wanted_mc_filled = [self.mc[key] for key in wanted_chain if key not in dashed_chain]
        for i in range(len(wanted_params)) :
            g = plots.get_single_plotter(width_inch=7, ratio = 1)
            g.settings.alpha_filled_add = 0.7
            g.settings.solid_contour_palefactor = 0.7
            g.settings.alpha_factor_contour_lines = 1
            g.plot_2d(wanted_mc_filled, x_param, wanted_params[i], filled = True, colors = colors)
            if len(dashed_chain) > 0 : 
                g.add_2d_contours(wanted_mc_dashed, x_param, wanted_params[i], ls='--', color = 'black')
            if legend is not None :
                g.add_legend(legend)
            plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(nbins=8))  # increase nbins
            plt.gca().yaxis.set_major_locator(ticker.MaxNLocator(nbins=8))

    def best_fit (self) :
        """
        Print out the best-fitting values of parameters from the chain
        using marginal mode and HPDI for uncertainty
        """
        for key in self.chain_files :
            print("\n" + "="*80)
            print("Best-fitting values of parameters from the chain: ", key)
            for i in range(len(self.sample_values_dict[key])) :
                ## Getting marginal mode
                values = self.sample_values_dict[key][i]
                index_weight = self.sample_names_dict[key].index("weight")
                weights = self.sample_values_dict[key][index_weight]
                hist, bin_edges = np.histogram(values,bins=200,weights=weights,density=True)
                bin_centers = 0.5*(bin_edges[1:]+bin_edges[:-1])
                mode = bin_centers[np.argmax(hist)]
                
                ## Getting the uncertainty using HPDI
                sort = np.argsort(values)
                sorted_values = values[sort]
                sorted_w = weights[sort]
                cdf = np.cumsum(sorted_w)
                intervals = []
                for j in range(len(weights)) :
                    target_cdf = cdf[j]+0.68
                    if target_cdf>1 :
                        break
                    target_index = np.argmin(np.abs(cdf-target_cdf))
                    interval = sorted_values[target_index] - sorted_values[j]
                    intervals.append(interval)
                interval = np.min(intervals)
                uncertainty = interval
        
                print(f"{self.sample_names_dict[key][i]} = {mode} +/- {uncertainty}")

In [50]:
files = {
        # "kids nl" : os.path.join(output_folder,"output_multinest_true.txt"),
        #  "kids lin" : os.path.join(output_folder,"output_multinest_kids_lin.txt"),
        #  "SN" : os.path.join(output_folder,"output_multinest_SN.txt"), 
        #  "SN+BAO" : os.path.join(output_folder,"output_multinest_SN_BAO.txt"), 
        #  "SN+BAO+QSO" : os.path.join(output_folder,"output_multinest_SN_BAO_QSO.txt"),
        #  "SN+BAO+QSO_z2" : os.path.join(output_folder,"output_multinest_SN_BAO_QSO_z2.txt"),
        #  "Marie" : os.path.join(output_folder,"output_multinest_SN_BAO_QSO_Marie.txt"),
          "WL" : os.path.join(output_folder,"output_multinest_WL_KIDS_VALUES.txt"),
         }

post = postprocess(files)
wanted_params = ['sigma_8', 's8', 'Sigma_8']
wanted_chain = [
                "kids lin", 
                "SN",
                "SN+BAO", 
                "SN+BAO+QSO", 
                "SN+BAO+QSO_z2", 
                "Marie",
                ]
dashed_chain = 'kids lin'
colors = ['red', 'green', 'blue', 'yellow', 'pink']
legend = ["SN","SN+BAO", "SN+BAO+QSO", "SN+BAO+QSO_z2", 'SN+BAO+QSO+CMB peaks', r'$\Lambda$CDM (linear)']

# post.plot_contour(wanted_params=wanted_params,
#                   x_param = 'om', 
#                   wanted_chain=wanted_chain, 
#                   dashed_chain=dashed_chain, 
#                   colors=colors, 
#                   legend=legend
#                 )
post.best_fit()

Removed no burn in

Best-fitting values of parameters from the chain:  WL
c0 = 1.4166690600528997 +/- 2.5173219932471143
c1 = 0.09674511238932615 +/- 0.6952156513254726
c2 = 0.025517857074737527 +/- 0.4576394995596407
c3 = 0.0871104557764068 +/- 0.6052567547078322
a = -3.0694231273346984 +/- 3.1913616760232806
uncorr_bias_1 = 0.3440466965413296 +/- 1.8655639267103652
uncorr_bias_2 = 0.15371851529495784 +/- 1.8230299407241546
uncorr_bias_3 = -0.9360662475686523 +/- 1.884254501924261
uncorr_bias_4 = -1.2773648020209407 +/- 1.9301959354493192
uncorr_bias_5 = 1.1037465654647745 +/- 1.9714150685376806
prior = -13.121498593203178 +/- 2.5335009376302065
like = -2287.8263314620253 +/- 1.4246088002121695
post = -2299.961487885803 +/- 3.2604345683341336
weight = 0.0038541532229032895 +/- 0.0023080516569034953
